# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KavyaR11/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)


**Lane:** engagement_fix \
**Task type:** Scoring (regression)

I'm framing this as scoring, not classification, even though the dataset
already contains a boolean `needs_engagement_fix` flag. A binary flag treats
all "bad" pages as equally bad, but a content team with limited hours needs to
know which flagged pages are the worst first. A continuous risk score lets a
strategist work down a ranked list instead of an undifferentiated flagged set.

In [41]:
# Import necessary libraries
from huggingface_hub import login
from google.colab import userdata

# Get the Hugging Face token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face Hub
login(token=hf_token)

print("Successfully logged in to Hugging Face Hub!")


Successfully logged in to Hugging Face Hub!


In [42]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("FlyRank/internship-lanes", "engagement_fix")
df = ds["train"].to_pandas()

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.head()


(33202, 31)
['client_hash_id', 'content_hash_id', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'client_has_gsc', 'client_has_ga4', 'impressions_30d', 'clicks_30d', 'pageviews_30d', 'sessions_30d', 'users_30d', 'engaged_sessions_30d', 'ai_sessions_30d', 'scroll_events_30d', 'ctr_30d', 'avg_position_30d', 'engagement_rate_30d', 'scroll_rate_30d', 'ai_traffic_pct_30d', 'impression_tier', 'position_tier', 'health_score', 'needs_indexing', 'is_quick_win', 'needs_ctr_fix', 'needs_engagement_fix', 'ai_opportunity']
client_hash_id           object
content_hash_id          object
content_type             object
main_intent              object
age_tier                 object
freshness_tier           object
word_count_tier          object
char_count_tier          object
client_has_gsc             bool
client_has_ga4             bool
impressions_30d           int64
clicks_30d                int64
pageviews_30d             int64
sessions_30d    

,client_hash_id,content_hash_id,content_type,main_intent,age_tier,freshness_tier,word_count_tier,char_count_tier,client_has_gsc,client_has_ga4,...,scroll_rate_30d,ai_traffic_pct_30d,impression_tier,position_tier,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity
0,client_f456856b7023ca82,content_24cff3eaee1df6bf,keyword article,informational,365+,31-90,None,None,True,True,...,8.33,0.00,moderate,striking,40,False,True,False,True,False
1,client_f456856b7023ca82,content_ee97ed5ba8d00f73,keyword article,informational,365+,31-90,None,None,True,True,...,22.92,0.00,good,page_1,60,False,False,True,True,True
2,client_f456856b7023ca82,content_de7992421a09be04,feedly article,unknown,181-365,31-90,1000-2000,<8000,True,True,...,4.55,0.00,low,top_3,55,False,False,False,True,False
3,client_f456856b7023ca82,content_79d5e5320dc338e8,feedly article,unknown,181-365,31-90,<1000,<8000,True,True,...,29.63,0.00,moderate,striking,55,False,False,False,True,True
4,client_f456856b7023ca82,content_27b5618b1038fc54,feedly article,unknown,181-365,31-90,1000-2000,<8000,True,True,...,23.08,8.33,moderate,page_1,60,False,False,False,True,False


## 2. Target or proxy

**Target/proxy:** `engagement_rate_30d` (inverted, so higher = more risk:
`risk_score = 1 - engagement_rate_30d`), optionally blended with
`scroll_rate_30d` since both capture different angles of "did the reader
actually engage."

This is a proxy, not the real target: true engagement failure means a reader's
intent wasn't met by the page. `engagement_rate_30d` (GA4's engaged-session
ratio) is an observable stand-in — a session can be short but fully satisfying
(quick answer found) or long but frustrated (reader hunting for something
they never found). GA4's definition of "engaged" doesn't fully separate these
cases, so my score inherits that blur.

Note: the dataset also has a pre-built rule-based flag, `needs_engagement_fix`.
I'm not using this as my target — I'm holding it out as a baseline to compare
against in Section 5.

In [43]:
df[['engagement_rate_30d', 'scroll_rate_30d', 'engaged_sessions_30d',
    'sessions_30d', 'needs_engagement_fix']].describe()

,engagement_rate_30d,scroll_rate_30d,engaged_sessions_30d,sessions_30d
count,33202.000000,33202.000000,33202.000000,33202.000000
mean,2.637504,6.227124,1.083368,44.879917
std,4.315668,7.104303,4.443106,1008.607189
min,0.000000,0.000000,0.000000,10.000000
25%,0.000000,0.000000,0.000000,15.000000
50%,0.000000,3.680000,0.000000,20.000000
75%,4.350000,9.380000,1.000000,36.000000
max,69.230000,29.790000,387.000000,128005.000000


## 3. Success metric

**Success metric:** Spearman rank correlation between my predicted risk score
and held-out `engagement_rate_30d` (inverted).

Ranking quality matters more than exact-value accuracy here — a strategist
works top-down through a prioritized list, so what matters is whether the
worst pages actually surface at the top, not whether my predicted number
matches the true rate to the decimal.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Unit of analysis

One row = one piece of content (`content_hash_id`), aggregated over a 30-day
window. Shown above: 5 real rows with the fields I'll use as features
(content_type, main_intent, age_tier, word_count_tier, session volume) and
the fields I'll build my target from (engagement_rate_30d, scroll_rate_30d).

In [45]:
unit_of_analysis = df[[
    'content_hash_id', 'content_type', 'main_intent', 'age_tier',
    'word_count_tier', 'sessions_30d', 'engaged_sessions_30d',
    'engagement_rate_30d', 'scroll_rate_30d', 'needs_engagement_fix'
]] .head(10)

unit_of_analysis



,content_hash_id,content_type,main_intent,age_tier,word_count_tier,sessions_30d,engaged_sessions_30d,engagement_rate_30d,scroll_rate_30d,needs_engagement_fix
0,content_24cff3eaee1df6bf,keyword article,informational,365+,None,10,1,10.00,8.33,True
1,content_ee97ed5ba8d00f73,keyword article,informational,365+,None,37,0,0.00,22.92,True
2,content_de7992421a09be04,feedly article,unknown,181-365,1000-2000,18,0,0.00,4.55,True
3,content_79d5e5320dc338e8,feedly article,unknown,181-365,<1000,52,0,0.00,29.63,True
4,content_27b5618b1038fc54,feedly article,unknown,181-365,1000-2000,12,0,0.00,23.08,True
5,content_6b45c4c8e7f14da3,feedly article,unknown,181-365,<1000,10,1,10.00,27.27,True
6,content_fa864a75fe3f5957,feedly article,unknown,181-365,<1000,34,0,0.00,26.32,True
7,content_0a700b0a8e5d5fad,keyword article,informational,365+,2000-3500,11,0,0.00,0.00,True
8,content_e3aae94881a927e2,keyword article,informational,365+,2000-3500,39,5,12.82,9.52,True
9,content_2fda1a779bb53428,keyword article,informational,365+,None,11,0,0.00,0.00,True


## 5. Why ML beats a fixed rule here

The dataset already contains a rule-based flag, `needs_engagement_fix` —
presumably a threshold like "engagement_rate_30d below X". A single threshold
can't account for **interactions**: a low engagement_rate on a reference/FAQ
page (main_intent may show quick lookups as normal) is different from the same
rate on a how-to guide expected to hold attention. Content_type, main_intent,
and word_count_tier likely shift what "normal" engagement looks like, and a
uniform rule flattens that context away.

I'm not assuming ML wins by default — I ran a quick group-by above to check
whether engagement_rate actually varies meaningfully across content_type/intent
within the same flag value. If it does, that's early evidence a rule ignoring
those fields is leaving signal on the table. Before building anything complex,
I plan to score my model's Spearman correlation against simply using
`needs_engagement_fix` as a ranking (flagged pages tied, unflagged tied) — if
ML doesn't clearly beat that trivial baseline, that's a real finding, not a
failure.

In [46]:
# Option B: min-max normalize, then invert
normalized = (df['engagement_rate_30d'] - df['engagement_rate_30d'].min()) / \
             (df['engagement_rate_30d'].max() - df['engagement_rate_30d'].min())
df['risk_score'] = 1 - normalized
df.groupby('needs_engagement_fix')['engagement_rate_30d'].describe()
df.groupby(['needs_engagement_fix', 'content_type'])['engagement_rate_30d'].mean()


needs_engagement_fix  content_type      
True                  comparison article    0.000000
                      feedly article        0.075383
                      keyword article       2.981991
Name: engagement_rate_30d, dtype: float64

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.